# 02 — Define a transfer split, match features, and learn normalization from training data

This notebook creates the exact prepared bundle consumed by the HPC model pipeline.

For each scenario it:

- loads CCLE and the platform matrices produced by notebook 01;
- designates datasets as **training** or **strictly held-out testing**;
- matches genes and proteins;
- fits RNA normalization parameters using training datasets only;
- applies those parameters unchanged to each held-out test dataset;
- harmonizes training protein targets by training domain;
- saves raw and normalized matrices, all learned parameters, manifests, and publication-ready plots.

During model hyperparameter CV, the training script refits the same normalization method inside each fold. The full-training parameters saved here are used only for the final per-protein fit and held-out prediction.

In [1]:
from pathlib import Path
import json
import sys
import joblib
import pandas as pd
from IPython.display import display, Image

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from prepare_scenario import prepare_scenario
from utils import load_yaml

print("PROJECT_ROOT:", PROJECT_ROOT)
print(load_yaml(PROJECT_ROOT / "configs" / "scenarios.yaml"))

PROJECT_ROOT: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\07.multidataset_transfer
{'scenarios': {'ccle_tmt_to_itraq_labelfree': {'train': ['CCLE', 'TMT'], 'test': ['iTRAQ', 'LabelFree'], 'description': 'Train once on CCLE + TMT and evaluate independently on iTRAQ and LabelFree.'}, 'ccle_tmt_itraq_to_labelfree': {'train': ['CCLE', 'TMT', 'iTRAQ'], 'test': ['LabelFree'], 'description': 'Leave LabelFree out.'}, 'ccle_tmt_labelfree_to_itraq': {'train': ['CCLE', 'TMT', 'LabelFree'], 'test': ['iTRAQ'], 'description': 'Leave iTRAQ out.'}, 'ccle_labelfree_itraq_to_tmt': {'train': ['CCLE', 'LabelFree', 'iTRAQ'], 'test': ['TMT'], 'description': 'Leave TMT out.'}}}


## Global scenario and normalization controls

In [2]:
# Edit only this cell, then run all cells.
SCENARIO = "ccle_tmt_to_itraq_labelfree"

# Leave these as None to use configs/scenarios.yaml.
# To define an explicit split in the notebook, provide lists and a unique OUTPUT_NAME.
TRAIN_DATASETS = None  # e.g. ["CCLE", "TMT"]
TEST_DATASETS = None   # e.g. ["iTRAQ", "LabelFree"]
OUTPUT_NAME = None     # e.g. "custom_ccle_tmt_to_itraq_labelfree"

# RNA choices: zscore, rank, binning, none
X_NORMALIZATION = "zscore"
# Protein target choices: domain_zscore (recommended), pooled_train_zscore, none
Y_NORMALIZATION = "domain_zscore"
N_BINS = 50
EPS = 1e-8
DOMAIN_BALANCED_NORMALIZER = True
QUANTILE_FIT_MAX_SAMPLES_PER_DOMAIN = 1000
MAKE_FIGURES = True
FORCE_REBUILD = False

## Match datasets and fit training-only transformations

In [3]:
prepared_dir = prepare_scenario(
    project_root=PROJECT_ROOT,
    scenario_name=SCENARIO,
    x_norm=X_NORMALIZATION,
    y_norm=Y_NORMALIZATION,
    n_bins=N_BINS,
    eps=EPS,
    domain_balanced_normalizer=DOMAIN_BALANCED_NORMALIZER,
    quantile_fit_max_samples_per_domain=QUANTILE_FIT_MAX_SAMPLES_PER_DOMAIN,
    make_figures=MAKE_FIGURES,
    force=FORCE_REBUILD,
    train_datasets_override=TRAIN_DATASETS,
    test_datasets_override=TEST_DATASETS,
    output_name=OUTPUT_NAME,
)
print("Prepared bundle:", prepared_dir)

FileNotFoundError: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\07.multidataset_transfer\data\processed\CCLE\transcriptomics.joblib

## Inspect dimensions, matching, and saved parameters

In [ ]:
manifest = json.loads((prepared_dir / "manifest.json").read_text())
summary = pd.read_csv(prepared_dir / "dataset_summary.tsv", sep="\t")
X_train_raw = joblib.load(prepared_dir / "train_X_raw.joblib")
X_train_norm = joblib.load(prepared_dir / "train_X_normalized.joblib")
Y_train_raw = joblib.load(prepared_dir / "train_Y_raw.joblib")
Y_train_norm = joblib.load(prepared_dir / "train_Y_normalized.joblib")
x_bundle = joblib.load(prepared_dir / "normalization" / "x_normalizer.joblib")
y_bundle = joblib.load(prepared_dir / "normalization" / "y_train_transformer.joblib")

display(summary)
print(json.dumps(manifest, indent=2))
print("train_X_raw:", X_train_raw.shape)
print("train_X_normalized:", X_train_norm.shape)
print("train_Y_raw:", Y_train_raw.shape)
print("train_Y_normalized:", Y_train_norm.shape)
print("X normalizer keys:", x_bundle.keys())
print("Y transformer domains:", list(y_bundle.get("params", {}).keys()))

for test_name in manifest["test_datasets"]:
    test_dir = prepared_dir / "tests" / test_name
    X_test = joblib.load(test_dir / "X_normalized.joblib")
    Y_test = joblib.load(test_dir / "Y_raw.joblib")
    print(f"{test_name}: X={X_test.shape}, Y={Y_test.shape}")

## Publication-ready diagnostic figures

In [ ]:
figure_dir = PROJECT_ROOT / "figures" / "preparation" / manifest["scenario"] / f"x-{X_NORMALIZATION}__y-{Y_NORMALIZATION}"
figure_files = sorted(figure_dir.glob("*.png"))
print("Figure directory:", figure_dir)
for figure in figure_files:
    print(figure.name)
    display(Image(filename=str(figure), width=950))

## Prepared outputs used by the HPC pipeline

The training scripts read this folder directly. Important files include:

- `train_X_raw.joblib`: used to refit normalization inside each CV fold;
- `train_X_normalized.joblib`: transformed with full-training parameters for the final model;
- `train_Y_raw.joblib` and `train_Y_normalized.joblib`;
- `normalization/x_normalizer.joblib` and `normalization/y_train_transformer.joblib`;
- `tests/<dataset>/X_normalized.joblib` and held-out protein measurements;
- `gene_universe.tsv`, `protein_universe.tsv`, and `manifest.json`.

No held-out test values contribute to normalization fitting, feature selection, hyperparameter selection, or model fitting.